# Tiny-GenImage Combined -> CommunityForensics-Eval Benchmark

Notebook này train 3 baseline trên **Tiny-GenImage combined** rồi evaluate zero-shot/cross-dataset trên **OwensLab/CommunityForensics-Eval / CompEval**.

```text
Train: Tiny-GenImage combined train split
Early stopping: Tiny-GenImage train_inner/val_inner
Internal sanity test: Tiny-GenImage validation split
External benchmark: CommunityForensics-Eval CompEval
```

CommunityForensics-Eval chỉ dùng để test cuối. Mặc định notebook lấy ngẫu nhiên 1000 ảnh bằng streaming shuffle với seed cố định, không dùng để train, không dùng để chọn epoch, không dùng để tune threshold.


## 0. Install Dependencies


In [ ]:
%pip install -q datasets open_clip_torch torchvision scikit-learn pandas tqdm


## 1. Imports And Project Root


In [ ]:
import copy
import gc
import io
import json
import random
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, IterableDataset
from torchvision.models import ResNet50_Weights, resnet18, resnet50
from tqdm.auto import tqdm

import open_clip


def find_code_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(kaggle_input.glob("*"))
        candidates.extend(kaggle_input.glob("*/*"))
        candidates.extend(kaggle_input.glob("*/*/*"))
        for init_file in kaggle_input.rglob("__init__.py"):
            if init_file.parent.name == "data_loader":
                return init_file.parent.parent

    for candidate in candidates:
        if (candidate / "data_loader" / "__init__.py").exists():
            return candidate

    print("/kaggle/input children:")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.glob("*")):
            print(" -", path)
    raise FileNotFoundError("Cannot find HoangHa_Code/data_loader.")


CODE_ROOT = find_code_root()
PROJECT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else CODE_ROOT
sys.path.insert(0, str(CODE_ROOT))
print("CODE_ROOT =", CODE_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)

from data_loader import (  # noqa: E402
    TinyGenImageKaggleConfig,
    TinyGenImageKaggleDataset,
    UnifiedSample,
    build_image_transform,
    build_kaggle_tiny_index,
    build_kaggle_tiny_splits,
    collate_unified_batch,
    find_tiny_genimage_root,
    summarize_index,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = torch.cuda.is_available()
USE_AMP = torch.cuda.is_available()
print("DEVICE =", DEVICE)


## 2. Config


In [ ]:
DATASET_ROOT = None

COMMFOR_DATASET_NAME = "OwensLab/CommunityForensics-Eval"
COMMFOR_SPLIT = "CompEval"
COMMFOR_STREAMING = True
MAX_COMMFOR_EVAL_SAMPLES = 1000  # random streaming sample from CompEval; set None for full CompEval
COMMFOR_SHUFFLE_BUFFER_SIZE = 100

RUN_BASELINES = [
    "clip_linear_head",
    "resnet50_last_layer",
    "npr_resnet18_from_scratch",
]

BALANCE_REAL = True
RANDOM_SEED = 42
VAL_FRACTION = 0.2
MAX_TRAIN_SAMPLES = None
MAX_TINY_INTERNAL_TEST_SAMPLES = None

BATCH_SIZE = 32
NUM_WORKERS = 0
MAX_EPOCHS = 10
PATIENCE = 3
MIN_DELTA = 1e-3
WEIGHT_DECAY = 1e-4

CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "openai"
HEAD_LR = 1e-3
NPR_LR = 2e-4

SAVE_PREDICTIONS = True
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "tiny_combined_to_commfor_eval"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
print("RUN_ID =", RUN_ID)


## 3. Utility Functions


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_run_dir(baseline_name: str) -> Path:
    run_dir = OUTPUT_ROOT / baseline_name / RUN_ID
    for subdir in ["checkpoints", "metrics", "predictions"]:
        (run_dir / subdir).mkdir(parents=True, exist_ok=True)
    return run_dir


def stratified_train_val_split(df: pd.DataFrame, val_fraction: float, seed: int):
    if len(df) < 4 or val_fraction <= 0:
        return df.reset_index(drop=True), df.reset_index(drop=True)

    stratify = df["label"].astype(str) + "_" + df["generator"].astype(str)
    if stratify.value_counts().min() < 2:
        stratify = df["label"]
    if pd.Series(stratify).value_counts().min() < 2:
        stratify = None

    train_df, val_df = train_test_split(
        df,
        test_size=val_fraction,
        random_state=seed,
        shuffle=True,
        stratify=stratify,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def compute_metrics(y_true, y_prob) -> dict[str, Any]:
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        metrics["average_precision"] = float(average_precision_score(y_true, y_prob))
    else:
        metrics["roc_auc"] = None
        metrics["average_precision"] = None
    return metrics


def evaluate_by_column(pred_df: pd.DataFrame, column: str) -> pd.DataFrame:
    rows = []
    if column not in pred_df.columns:
        return pd.DataFrame(rows)
    for value, part in pred_df.groupby(column):
        if len(part) == 0:
            continue
        metrics = compute_metrics(part["label"].to_numpy(), part["fake_probability"].to_numpy())
        metrics[column] = value
        metrics["num_samples"] = int(len(part))
        rows.append(metrics)
    return pd.DataFrame(rows).sort_values(column) if rows else pd.DataFrame(rows)


def save_json(path: Path, data: dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 4. Tiny-GenImage Combined Loaders


In [ ]:
def build_tiny_combined_splits(detected_root: Path) -> dict[str, Any]:
    config = TinyGenImageKaggleConfig(
        dataset_root=str(detected_root),
        eval_case="combined",
        balance_real=BALANCE_REAL,
        seed=RANDOM_SEED,
        max_train_samples=MAX_TRAIN_SAMPLES,
        max_eval_samples=MAX_TINY_INTERNAL_TEST_SAMPLES,
    )
    splits = build_kaggle_tiny_splits(config)
    print(splits["notes"])
    print("train real/fake:", splits["train_real_count"], splits["train_fake_count"])
    print("tiny test real/fake:", splits["eval_real_count"], splits["eval_fake_count"])
    return splits


def build_tiny_loaders(splits: dict[str, Any], transform_train, transform_eval):
    train_inner_df, val_inner_df = stratified_train_val_split(
        splits["train_df"],
        val_fraction=VAL_FRACTION,
        seed=RANDOM_SEED,
    )
    tiny_test_df = splits["eval_df"].reset_index(drop=True)
    print("inner train/val:", len(train_inner_df), len(val_inner_df))
    print("tiny internal test:", len(tiny_test_df))

    train_dataset = TinyGenImageKaggleDataset(train_inner_df, eval_case="combined", transform=transform_train)
    val_dataset = TinyGenImageKaggleDataset(val_inner_df, eval_case="combined", transform=transform_eval)
    tiny_test_dataset = TinyGenImageKaggleDataset(tiny_test_df, eval_case="combined", transform=transform_eval)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_unified_batch,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_unified_batch,
    )
    tiny_test_loader = DataLoader(
        tiny_test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_unified_batch,
    )
    return train_loader, val_loader, tiny_test_loader, train_inner_df, val_inner_df, tiny_test_df


seed_everything(RANDOM_SEED)
detected_tiny_root = find_tiny_genimage_root(DATASET_ROOT)
print("Detected Tiny-GenImage root:", detected_tiny_root)

index_df = build_kaggle_tiny_index(TinyGenImageKaggleConfig(dataset_root=str(detected_tiny_root)))
summary_path = OUTPUT_ROOT / f"tiny_structure_summary_{RUN_ID}.csv"
summarize_index(index_df).to_csv(summary_path, index=False)
print("Tiny structure summary saved:", summary_path)
print("Generators:", sorted(index_df["generator"].unique().tolist()))

TINY_SPLITS = build_tiny_combined_splits(detected_tiny_root)


## 5. CommunityForensics-Eval Loader


In [ ]:
def image_from_commfor_record(record: dict[str, Any]) -> Image.Image:
    image_data = record.get("image_data", record.get("image"))
    if isinstance(image_data, Image.Image):
        return image_data.convert("RGB")
    if isinstance(image_data, (bytes, bytearray)):
        return Image.open(io.BytesIO(image_data)).convert("RGB")
    if isinstance(image_data, dict):
        if image_data.get("bytes") is not None:
            return Image.open(io.BytesIO(image_data["bytes"])).convert("RGB")
        if image_data.get("path") is not None:
            return Image.open(image_data["path"]).convert("RGB")
    if isinstance(image_data, list):
        return Image.open(io.BytesIO(bytes(image_data))).convert("RGB")
    raise TypeError(f"Unsupported image_data type: {type(image_data)}")


class CommunityForensicsEvalIterableDataset(IterableDataset):
    def __init__(self, hf_dataset, transform=None, eval_case="cross_dataset_commfor_eval"):
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.eval_case = eval_case

    def __iter__(self):
        for index, record in enumerate(self.hf_dataset):
            image = image_from_commfor_record(record)
            if self.transform is not None:
                image = self.transform(image)

            label = int(record.get("label"))
            model_name = str(record.get("model_name") or record.get("architecture") or "unknown")
            sample = UnifiedSample(
                sample_id=str(record.get("image_name") or f"commfor_eval:{index}"),
                label=label,
                label_name="fake" if label == 1 else "real",
                dataset_source=COMMFOR_DATASET_NAME,
                generator=model_name,
                split=str(record.get("split") or COMMFOR_SPLIT),
                eval_case=self.eval_case,
                prompt=record.get("prompt"),
                metadata={
                    "image_name": record.get("image_name"),
                    "format": record.get("format"),
                    "resolution": record.get("resolution"),
                    "mode": record.get("mode"),
                    "model_name": record.get("model_name"),
                    "architecture": record.get("architecture"),
                    "real_source": record.get("real_source"),
                    "subset": record.get("subset"),
                    "nsfw_flag": record.get("nsfw_flag"),
                },
            ).as_dict()
            sample["image_name"] = record.get("image_name")
            sample["model_name"] = record.get("model_name")
            sample["architecture"] = record.get("architecture")
            sample["real_source"] = record.get("real_source")
            sample["subset"] = record.get("subset")
            sample["nsfw_flag"] = record.get("nsfw_flag")
            sample["image"] = image
            yield sample


def build_commfor_eval_loader(transform_eval):
    hf_dataset = load_dataset(
        COMMFOR_DATASET_NAME,
        split=COMMFOR_SPLIT,
        streaming=COMMFOR_STREAMING,
    )

    if COMMFOR_STREAMING:
        hf_dataset = hf_dataset.shuffle(
            seed=RANDOM_SEED,
            buffer_size=COMMFOR_SHUFFLE_BUFFER_SIZE,
        )
        if MAX_COMMFOR_EVAL_SAMPLES is not None:
            hf_dataset = hf_dataset.take(MAX_COMMFOR_EVAL_SAMPLES)
    elif MAX_COMMFOR_EVAL_SAMPLES is not None:
        hf_dataset = hf_dataset.shuffle(seed=RANDOM_SEED).select(range(MAX_COMMFOR_EVAL_SAMPLES))

    dataset = CommunityForensicsEvalIterableDataset(hf_dataset, transform=transform_eval)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_unified_batch,
    )


## 6. Model Definitions


In [ ]:
class ClipLinearHead(nn.Module):
    def __init__(self, clip_model, embed_dim: int, num_classes: int = 2):
        super().__init__()
        self.clip_model = clip_model
        self.head = nn.Linear(embed_dim, num_classes)
        for param in self.clip_model.parameters():
            param.requires_grad = False
        self.clip_model.eval()

    def train(self, mode: bool = True):
        super().train(mode)
        self.clip_model.eval()
        return self

    def forward(self, images):
        with torch.no_grad():
            features = self.clip_model.encode_image(images)
            features = F.normalize(features.float(), dim=-1)
        return self.head(features)


def make_clip_linear_head(device: str):
    clip_model, _, preprocess = open_clip.create_model_and_transforms(
        CLIP_MODEL_NAME,
        pretrained=CLIP_PRETRAINED,
        device=device,
    )
    clip_model.eval()
    embed_dim = int(getattr(clip_model.visual, "output_dim", 512))
    model = ClipLinearHead(clip_model=clip_model, embed_dim=embed_dim).to(device)
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    return model, optimizer, preprocess, preprocess, "clip_linear_head.pt"


class FrozenResNet50LastLayer(nn.Module):
    def __init__(self):
        super().__init__()
        weights = ResNet50_Weights.DEFAULT
        self.backbone = resnet50(weights=weights)
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 2)

    def train(self, mode: bool = True):
        super().train(mode)
        for name, module in self.backbone.named_children():
            if name == "fc":
                module.train(mode)
            else:
                module.eval()
        return self

    def forward(self, images):
        return self.backbone(images)


def make_resnet50_last_layer(device: str):
    model = FrozenResNet50LastLayer().to(device)
    optimizer = torch.optim.AdamW(model.backbone.fc.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    transform = build_image_transform(image_size=224, train=True)
    eval_transform = build_image_transform(image_size=224, train=False)
    return model, optimizer, transform, eval_transform, "resnet50_last_layer.pt"


class NPRLayer(nn.Module):
    def __init__(self, factor=0.5, scale=2.0 / 3.0):
        super().__init__()
        self.factor = factor
        self.scale = scale

    def forward(self, x):
        _, _, height, width = x.shape
        if height % 2 == 1:
            x = x[:, :, :-1, :]
        if width % 2 == 1:
            x = x[:, :, :, :-1]
        down = F.interpolate(x, scale_factor=self.factor, mode="nearest", recompute_scale_factor=True)
        up = F.interpolate(down, size=x.shape[-2:], mode="nearest")
        return (x - up) * self.scale


class NPRResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.npr = NPRLayer()
        self.backbone = resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x):
        return self.backbone(self.npr(x))


def make_npr_resnet18(device: str):
    model = NPRResNet18(num_classes=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=NPR_LR, weight_decay=WEIGHT_DECAY)
    transform = build_image_transform(image_size=224, train=True)
    eval_transform = build_image_transform(image_size=224, train=False)
    return model, optimizer, transform, eval_transform, "npr_resnet18_from_scratch.pt"


BASELINE_FACTORIES = {
    "clip_linear_head": make_clip_linear_head,
    "resnet50_last_layer": make_resnet50_last_layer,
    "npr_resnet18_from_scratch": make_npr_resnet18,
}


## 7. Train And Evaluation Helpers


In [ ]:
def copy_state_dict_to_cpu(model: nn.Module) -> dict[str, torch.Tensor]:
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def make_checkpoint_state(model: nn.Module, baseline_name: str) -> dict[str, Any]:
    if baseline_name == "clip_linear_head":
        return {
            "head": {key: value.detach().cpu().clone() for key, value in model.head.state_dict().items()},
            "checkpoint_type": "clip_linear_head_only",
        }
    return {
        "model": copy_state_dict_to_cpu(model),
        "checkpoint_type": "full_model",
    }


def load_checkpoint_state(model: nn.Module, checkpoint: dict[str, Any]) -> None:
    if checkpoint.get("checkpoint_type") == "clip_linear_head_only":
        model.head.load_state_dict(checkpoint["head"])
    else:
        model.load_state_dict(checkpoint["model"])


def train_one_epoch(model, loader, optimizer, criterion, scaler) -> float:
    model.train()
    losses = []
    for batch in tqdm(loader, desc="train", leave=False):
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else 0.0


@torch.no_grad()
def predict(model, loader, desc="eval"):
    model.eval()
    labels, probs, rows = [], [], []
    for batch in tqdm(loader, desc=desc, leave=False):
        images = batch["image"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(images)
        prob_fake = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
        probs.extend(prob_fake.tolist())
        labels.extend(batch["label"].numpy().tolist())
        rows.extend(batch["metadata"])
    return np.array(labels, dtype=int), np.array(probs, dtype=float), pd.DataFrame(rows)


def train_with_early_stopping(model, optimizer, train_loader, val_loader, checkpoint_path: Path, baseline_name: str):
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best_metric = -float("inf")
    best_epoch = 0
    best_state = None
    bad_epochs = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        y_val, p_val, _ = predict(model, val_loader, desc="tiny val")
        val_metrics = compute_metrics(y_val, p_val)
        current = val_metrics["balanced_accuracy"]
        history.append({"epoch": epoch, "train_loss": loss, **val_metrics})
        print(f"epoch {epoch}/{MAX_EPOCHS} loss={loss:.4f} val_bal_acc={current:.4f}")

        if current > best_metric + MIN_DELTA:
            best_metric = current
            best_epoch = epoch
            best_state = make_checkpoint_state(model, baseline_name)
            torch.save(best_state, checkpoint_path)
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"early stopping at epoch {epoch}; best_epoch={best_epoch}")
                break

    if best_state is None:
        best_state = make_checkpoint_state(model, baseline_name)
        torch.save(best_state, checkpoint_path)
    load_checkpoint_state(model, torch.load(checkpoint_path, map_location=DEVICE))
    return pd.DataFrame(history), best_epoch, best_metric


def evaluate_and_save(model, loader, run_dir: Path, baseline_name: str, dataset_tag: str):
    y_true, y_prob, meta_df = predict(model, loader, desc=dataset_tag)
    pred_df = meta_df.copy()
    pred_df["label"] = y_true
    pred_df["predicted_label"] = (y_prob >= 0.5).astype(int)
    pred_df["fake_probability"] = y_prob
    pred_df["baseline_name"] = baseline_name
    pred_df["dataset_tag"] = dataset_tag

    metrics = compute_metrics(y_true, y_prob)
    metrics.update({
        "baseline_name": baseline_name,
        "dataset_tag": dataset_tag,
        "num_samples": int(len(pred_df)),
        "threshold": 0.5,
        "train_dataset": "Tiny-GenImage",
        "train_eval_case": "combined",
        "external_dataset": COMMFOR_DATASET_NAME if dataset_tag == "community_forensics_eval" else "Tiny-GenImage",
    })

    save_json(run_dir / "metrics" / f"{dataset_tag}_overall_metrics.json", metrics)
    by_generator = evaluate_by_column(pred_df, "generator")
    if len(by_generator):
        by_generator.to_csv(run_dir / "metrics" / f"{dataset_tag}_generator_metrics.csv", index=False)
    by_architecture = evaluate_by_column(pred_df, "architecture")
    if len(by_architecture):
        by_architecture.to_csv(run_dir / "metrics" / f"{dataset_tag}_architecture_metrics.csv", index=False)
    if SAVE_PREDICTIONS:
        pred_df.to_csv(run_dir / "predictions" / f"{dataset_tag}_predictions.csv", index=False)
    return metrics


## 8. Run Three Baselines


In [ ]:
def run_baseline(baseline_name: str) -> dict[str, Any]:
    if baseline_name not in BASELINE_FACTORIES:
        raise ValueError(f"Unknown baseline: {baseline_name}")

    print("\n" + "=" * 80)
    print("Running baseline:", baseline_name)
    print("=" * 80)

    seed_everything(RANDOM_SEED)
    run_dir = make_run_dir(baseline_name)
    model, optimizer, transform_train, transform_eval, checkpoint_name = BASELINE_FACTORIES[baseline_name](DEVICE)
    checkpoint_path = run_dir / "checkpoints" / checkpoint_name

    train_loader, val_loader, tiny_test_loader, train_inner_df, val_inner_df, tiny_test_df = build_tiny_loaders(
        TINY_SPLITS,
        transform_train=transform_train,
        transform_eval=transform_eval,
    )
    train_inner_df.to_csv(run_dir / "metrics" / "tiny_train_inner_split.csv", index=False)
    val_inner_df.to_csv(run_dir / "metrics" / "tiny_val_inner_split.csv", index=False)
    tiny_test_df.to_csv(run_dir / "metrics" / "tiny_internal_test_split.csv", index=False)

    checkpoint_kind = "clip_linear_head_only" if baseline_name == "clip_linear_head" else "full_model"
    config_summary = {
        "baseline_name": baseline_name,
        "run_id": RUN_ID,
        "train_dataset": "Tiny-GenImage",
        "train_eval_case": "combined",
        "balance_real": BALANCE_REAL,
        "random_seed": RANDOM_SEED,
        "max_train_samples": MAX_TRAIN_SAMPLES,
        "max_tiny_internal_test_samples": MAX_TINY_INTERNAL_TEST_SAMPLES,
        "max_commfor_eval_samples": MAX_COMMFOR_EVAL_SAMPLES,
        "commfor_shuffle_buffer_size": COMMFOR_SHUFFLE_BUFFER_SIZE,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "min_delta": MIN_DELTA,
        "weight_decay": WEIGHT_DECAY,
        "head_lr": HEAD_LR,
        "npr_lr": NPR_LR,
        "device": DEVICE,
        "use_amp": USE_AMP,
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_kind": checkpoint_kind,
        "tiny_train_rows": int(len(TINY_SPLITS["train_df"])),
        "tiny_test_rows": int(len(TINY_SPLITS["eval_df"])),
    }
    save_json(run_dir / "metrics" / "config.json", config_summary)

    history_df, best_epoch, best_val_bal_acc = train_with_early_stopping(
        model,
        optimizer,
        train_loader,
        val_loader,
        checkpoint_path,
        baseline_name=baseline_name,
    )
    history_df.to_csv(run_dir / "metrics" / "history.csv", index=False)

    print("Evaluating Tiny-GenImage validation split for sanity check...")
    tiny_metrics = evaluate_and_save(
        model,
        tiny_test_loader,
        run_dir=run_dir,
        baseline_name=baseline_name,
        dataset_tag="tiny_genimage_validation",
    )

    print("Evaluating CommunityForensics-Eval CompEval...")
    commfor_loader = build_commfor_eval_loader(transform_eval)
    commfor_metrics = evaluate_and_save(
        model,
        commfor_loader,
        run_dir=run_dir,
        baseline_name=baseline_name,
        dataset_tag="community_forensics_eval",
    )

    result = {
        **config_summary,
        "best_epoch": best_epoch,
        "best_val_balanced_accuracy": best_val_bal_acc,
        "tiny_balanced_accuracy": tiny_metrics.get("balanced_accuracy"),
        "tiny_f1": tiny_metrics.get("f1"),
        "tiny_roc_auc": tiny_metrics.get("roc_auc"),
        "commfor_balanced_accuracy": commfor_metrics.get("balanced_accuracy"),
        "commfor_f1": commfor_metrics.get("f1"),
        "commfor_roc_auc": commfor_metrics.get("roc_auc"),
        "commfor_average_precision": commfor_metrics.get("average_precision"),
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_kind": checkpoint_kind,
    }
    save_json(run_dir / "metrics" / "summary.json", result)

    del train_loader, val_loader, tiny_test_loader, commfor_loader, model, optimizer
    cleanup_cuda()
    return result


all_results = []
for baseline_name in RUN_BASELINES:
    all_results.append(run_baseline(baseline_name))

summary_df = pd.DataFrame(all_results)
summary_path = OUTPUT_ROOT / f"summary_tiny_combined_to_commfor_eval_{RUN_ID}.csv"
manifest_path = OUTPUT_ROOT / f"checkpoint_manifest_{RUN_ID}.csv"
summary_df.to_csv(summary_path, index=False)
summary_df[[
    "baseline_name",
    "checkpoint_path",
    "checkpoint_kind",
    "best_epoch",
    "best_val_balanced_accuracy",
    "tiny_balanced_accuracy",
    "commfor_balanced_accuracy",
    "commfor_roc_auc",
    "commfor_average_precision",
]].to_csv(manifest_path, index=False)
print("Saved summary:", summary_path)
print("Saved checkpoint manifest:", manifest_path)
display(summary_df[[
    "baseline_name",
    "checkpoint_path",
    "checkpoint_kind",
    "best_epoch",
    "best_val_balanced_accuracy",
    "tiny_balanced_accuracy",
    "tiny_roc_auc",
    "commfor_balanced_accuracy",
    "commfor_roc_auc",
    "commfor_average_precision",
]])


## 9. Notes For Report

Protocol này nên được mô tả là **cross-dataset external benchmark**:

```text
Models are trained only on Tiny-GenImage under the combined setting. The CommunityForensics-Eval CompEval split is used only for final zero-shot evaluation and is never used for training, early stopping, or threshold tuning.
```

Nên báo cáo cả metric tổng và metric theo `generator/model_name`, vì CommunityForensics-Eval có nhiều model sinh ảnh khác nhau.
